In [1]:
from torch.utils.data import Dataset
from PIL import Image
import torch

from transformers import AutoImageProcessor
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoModelForImageClassification
import numpy as np
from transformers import TrainingArguments, Trainer

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1: Pre processing

Read in the train images csv file and create an auto image processor that will be used to convert the images from the dataset into pixel values

In [34]:
df = pd.read_csv("../data/train_images.csv")

In [35]:
MODEL_NAME = "google/vit-base-patch16-224"

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

class ImageDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open('../data' + row["image_path"]).convert("RGB")

        pixel_values = processor(image, return_tensors="pt")["pixel_values"].squeeze()

        return {
            "pixel_values": pixel_values,
            "labels": torch.tensor(row["label"])
        }


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


## 2: Training

Here we split the data into a training and validation dataset, ensuring we stratify by label to have a similar distribution of labels in both splits.

We then load the hugging face pre-trained model and train it using the hugging face trainer utility



In [36]:
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["label"])

train_dataset = ImageDataset(train_df)
val_dataset = ImageDataset(val_df)

In [37]:

model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(df['label'].unique()),
    ignore_mismatched_sizes=True
)

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([200]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([200, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [38]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    learning_rate=5e-5,
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

/Users/apple/.pyenv/versions/aml-feathers-3-12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

# 3: Run inference on test dataset

We create predictions on the test dataset and store it to a csv file

In [10]:
test_df = pd.read_csv("../data/test_images_path.csv")
test_dataset = ImageDataset(test_df)
predictions = trainer.predict(test_dataset)
pred_classes = np.argmax(predictions.predictions, axis=1)

In [16]:
predictions_df = pd.DataFrame(columns=["label"], data=pred_classes)
predictions_df['id'] = predictions_df.index + 1
predictions_df.to_csv("baseline_results.csv", index=False)